# 2. Service-to-Service with Client Credentials

**Scenario**: a background daemon (cron job, worker, scheduled function) needs to call an API. There's no user sitting in front of a browser. This is the canonical *service-to-service* (S2S) pattern.

The flow is called **client credentials**:

```
daemon ──(client_id + client_secret)──▶ Entra ID  ──▶ access token
daemon ──(Bearer token)─────────────────▶ api-b    ──▶ data
```

Entra treats the daemon like any other principal. It checks:
1. Does the secret match?
2. Has the daemon's service principal been **granted** the app-role it's asking for, on the target API?

The token it issues carries the granted roles in the `roles` claim.

## The `/.default` scope

When you do client credentials, you can't pick individual scopes — you ask for `<resource>/.default`, meaning *"give me everything this app has been granted on this resource"*. That's an Entra-ism: delegated scopes need user consent, so for the app-only case the `/.default` trick hands back all pre-consented app-roles.

In [ ]:
import httpx, json, base64

TOKEN_URL = 'http://localhost:9000/contoso/oauth2/v2.0/token'
API_B     = 'http://localhost:8002'

def decode_payload(t):
    p = t.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(p + '=' * (-len(p) % 4)))

# --- 1. Get a token as the daemon app ---
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-b/.default',
})
r.raise_for_status()
token = r.json()['access_token']
print(json.dumps(decode_payload(token), indent=2))

Notice:
- `aud = api://api-b` — token is *for* API-B, not usable elsewhere.
- `roles = ['Files.Read.All']` — the app role granted to the daemon.
- **No `upn` / `scp`** — this is an app-only token; no user was involved.

## 2. Call API-B

In [ ]:
r = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {token}'})
print(r.status_code)
print(json.dumps(r.json(), indent=2))

API-B's handler checks `roles` and returns *all* files because the daemon is acting on its own (app-only mode).

## 3. What happens if we don't have the app role?

Let's simulate a daemon without the grant by manually issuing a token without roles. The mock Entra only grants roles listed in `granted_app_roles`, so if we use an app that has none granted on api-b (like `api-a-client-id` used as a pure daemon without pre-grants… actually api-a *does* have the role). Instead: use an invalid secret to show the auth failure.

In [ ]:
# Wrong secret
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'WRONG',
    'scope': 'api://api-b/.default',
})
print('status:', r.status_code, r.json())

## 4. What happens with a token for the wrong API?

Tokens include `aud` (audience). API-B rejects anything that isn't `api://api-b`.

In [ ]:
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-a/.default',   # wrong target
})
other = r.json().get('access_token')
r2 = httpx.get(f'{API_B}/files', headers={'Authorization': f'Bearer {other}'})
print('status:', r2.status_code, r2.json())

That's the whole point of `aud` — a stolen token from one API can't be replayed against another.

## 5. Using MSAL (what you'd actually write in production)

Nobody POSTs to the token endpoint by hand in real code — you use Microsoft's **MSAL** library. It handles caching, refresh, retries, certificate auth, etc.

In [ ]:
from msal import ConfidentialClientApplication

app = ConfidentialClientApplication(
    client_id='daemon-client-id',
    client_credential='daemon-secret-value',
    authority='http://localhost:9000/contoso',
    validate_authority=False,   # only needed for our mock; skip in real code
)
result = app.acquire_token_for_client(scopes=['api://api-b/.default'])
print('got token:', 'access_token' in result)
print('expires in', result.get('expires_in'), 'seconds')

In real Azure you'd use:
```python
ConfidentialClientApplication(
    client_id=os.environ['AZURE_CLIENT_ID'],
    client_credential=os.environ['AZURE_CLIENT_SECRET'],
    authority=f'https://login.microsoftonline.com/{tenant_id}',
)
```

Even better — don't use a secret at all. **Managed identity** (notebook 3) removes the need entirely.

## Summary

- **Client credentials** = app authenticates with its own creds, gets app-only token.
- Token carries `roles`, *not* `scp` — these are application permissions.
- Always call `<resource>/.default` for client credentials.
- Audience (`aud`) scoping stops token replay across APIs.
- Use MSAL, not raw HTTP, in real code.